# Loop0, stage 02: apply the gating strategy

The three polygons in `gates.npz` are applied to every cell.  A cell is kept only
when it is inside all three, which is the sequential gating strategy: a cell, a
single cell, and alive.

## Inputs

| file | produced by | holds |
|---|---|---|
| `Loop0.h5ad` | stage 00 | every cell of every well |
| `gates.npz` | stage 01 of this loop | the three gate polygons |

## Outputs

| file | holds |
|---|---|
| `Loop0_Gated.h5ad` | every cell that passed all three gates |
| `Loop0_Gated_500k.h5ad` | about 500,000 of those cells |
| `Loop0_Gated_ExpRem.h5ad` | the gated cells, without the experiments listed below |
| `Loop0_Gated_ExpRem_500k.h5ad` | about 500,000 of those cells |

Stage 04 reads `Loop0_Gated_ExpRem.h5ad`.

Run the cells from top to bottom.

In [ ]:
import matplotlib.pyplot as plt
import pytometry as pm
import scanpy as sc

from labcompass import apply_saved_gates, downsample, target_size_for

IN_ALL = "Loop0.h5ad"
GATE_FILE = "gates.npz"

OUT_GATED = "Loop0_Gated.h5ad"
OUT_GATED_SUBSET = "Loop0_Gated_500k.h5ad"

DESIRED_CELL_NUMBER = 500000
DOWNSAMPLE_SEED = 0

In [ ]:
adata_all = sc.read(IN_ALL)
print(f"{adata_all.n_obs:,} cells before gating")
adata_all

## Apply the gates

In [ ]:
before = adata_all.n_obs
adata_all, masks = apply_saved_gates(adata_all, GATE_FILE, mode="all")

for i, mask in enumerate(masks, start=1):
    print(f"gate {i} alone keeps {mask.sum():,} of {before:,} cells")
print(f"all three together keep {adata_all.n_obs:,} cells "
      f"({100 * adata_all.n_obs / before:.1f} per cent)")

The live gate is drawn against 7-AAD.  After gating, the bright events should be gone.

In [ ]:
pm.pl.scatter_density(
    adata_all, x="FSC-A :: FSC - Area", y="7-AAD",
    x_lim=[0, 1_500_000], y_lim=[-10_000, 10_000],
)

In [ ]:
plt.close("all")

In [ ]:
adata_all.write(OUT_GATED)
print(f"wrote {OUT_GATED}: {adata_all.n_obs:,} cells")

## Cells kept per experiment

In [ ]:
counts = adata_all.obs["experiment_number"].value_counts()
ax = counts.plot.bar(figsize=(max(6, 0.2 * len(counts)), 4))
ax.set_ylabel("cells kept")
ax.set_xlabel("experiment number")
plt.tight_layout()
plt.show()
counts

## Write the downsampled gated object

In [ ]:
target_size = target_size_for(adata_all, DESIRED_CELL_NUMBER, obs="experiment_number")
print(f"cap of {target_size:,} cells per experiment")
downsample(
    adata_all, obs="experiment_number", target_size=target_size, seed=DOWNSAMPLE_SEED
).write(OUT_GATED_SUBSET)
print(f"wrote {OUT_GATED_SUBSET}")

## Drop the experiments that are not part of the screen

These experiments were acquired on the same plates but do not belong to the
analysis.  The reason for each block is written next to it.  Dropping them here
rather than later keeps every downstream file consistent.

The entry `"FMO"` is kept for the case where an experiment number is the text
`FMO`.  It matches nothing when the experiment numbers are integers.

In [ ]:
EXPERIMENTS_TO_REMOVE = [
    1, 2, 3, 4, 5, 6, 7, 8, 9,
    10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
    20, 21, 22, 23, 24, 25, 26, 27, 28, 29,
    30, 31, 32, 33, 34, 35, 36, 37, 38, 39,  # LPHO01 diff voltages
    44, 45, 46, 47, 48, 49, 50, 51, 52,
    56, 57, 58, 59, 60, 61, 62, 63, 64,
    68, 69, 70, 71, 72, 73, 74, 75, 76,
    80, 81, 82, 83, 84, 85, 86, 87, 88,  # LPHO03a remove, GMCSF different timepoints
    89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99,
    100, 101, 102, 103, 104,
    167, 168, 169, 170, 171, 172, 173, 174,
    175, 176, 177, 178, 179, 180, 181, 182,  # LPHO03b day 14 only, aggresive bleeding
    124, 132,  # LPHO04b, day 27 exp only
    134, 135, 136, 137,  # LPHo04 hydrogels
    148, 149, 150, 151, 152,  # LPHO05 hydrogels
    153, 154, 155, 156,  # JAR03 LARRY cells
    165, 166,  # JAR03 hydrogels
    186, 187,  # replicates of 115 and 116
    190, 191, 192, 193, 194, 195, 196, 197,  # CB2 diff seeding
    198, 199,  # hydrogels cb2 diff seeding
    200, 201, 202, 203, 204,  # JAR04 diff voltages
    211, 212, 213, 214,  # seeding experiment
    215,  # few cells
    1167, 1168,  # duplicates of 1089 and 1090
    "FMO",
]

In [ ]:
before = adata_all.n_obs
adata_all = adata_all[
    ~adata_all.obs["experiment_number"].isin(EXPERIMENTS_TO_REMOVE)
].copy()
print(f"{before:,} cells -> {adata_all.n_obs:,} cells, "
      f"{adata_all.obs['experiment_number'].nunique()} experiments left")

adata_all.write("Loop0_Gated_ExpRem.h5ad")
target_size = target_size_for(adata_all, DESIRED_CELL_NUMBER, obs="experiment_number")
print(f"cap of {target_size:,} cells per experiment")
downsample(
    adata_all, obs="experiment_number", target_size=target_size, seed=DOWNSAMPLE_SEED
).write("Loop0_Gated_ExpRem_500k.h5ad")
print("wrote Loop0_Gated_ExpRem.h5ad and Loop0_Gated_ExpRem_500k.h5ad")